*0.1 Python for GenAI*

# virtual envs

**The situation.** Two projects on one laptop. Project A needs `openai` version 1; project B needs version 2. Both were installed into the system Python. Whichever was installed last wins. The other project now fails with an import error that looks exactly like a bug in the code.

**The fix: a private Python per project.** A *virtual environment* is a folder — usually `.venv` — with its own copy of the interpreter and its own installed packages. Project A's libraries and project B's never meet. uv and Poetry create it for you; you rarely touch it directly.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Where is this notebook running?** `sys.prefix` is where the running Python keeps its packages; `sys.base_prefix` is the original installation. If they differ, you are inside a virtual environment.

In [2]:
import sys
from importlib import metadata
from pathlib import Path

print("interpreter:", sys.executable.replace(str(Path.home()), "~"))
print("inside a virtual environment:", sys.prefix != sys.base_prefix)
print("openai version here:", metadata.version("openai"))
assert sys.prefix != sys.base_prefix

interpreter: ~/Documents/GenAI/GenAI_Learnings/GenAILayers/.venv/bin/python3
inside a virtual environment: True
openai version here: 2.54.0


**A brand-new environment has nothing in it.** Create one and try to import `openai` there.

In [3]:
import subprocess
import tempfile

with tempfile.TemporaryDirectory() as folder:
    venv = Path(folder) / ".venv"
    subprocess.run([sys.executable, "-m", "venv", str(venv)], check=True)
    probe = subprocess.run(
        [str(venv / "bin" / "python"), "-c", "import openai"], capture_output=True, text=True
    )
    print("fresh environment, import openai →", probe.stderr.strip().splitlines()[-1])
assert "No module named 'openai'" in probe.stderr

fresh environment, import openai → ModuleNotFoundError: No module named 'openai'


**Reading the output.** This notebook runs inside `.venv` with `openai` installed. The fresh environment has no third-party packages at all — until a lock file installs them.

```
system Python (untouched)
   ├── project A/.venv   openai 1.x
   └── project B/.venv   openai 2.x        never in the same folder, never in conflict
```

| Use it when | Don't when | Instead use |
|---|---|---|
| always — one per project | never install into the system Python | Docker as the environment; conda for native scientific stacks |

**Watch out**
- "Module not found" for something you just installed almost always means the editor or terminal is using a different Python. Check `sys.executable`.
- Build the environment inside the Docker image from the lock file; never `pip install` when the container starts.
- The operating system depends on its own Python; installing into it can break the OS.